In [1]:
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, avg, lag, row_number, mean, stddev, sum, year, month

In [2]:
full_grouped_file_path = "hdfs:///data/covid/staging/full_grouped"
# worldometer_data_file_path = "hdfs:///data/covid/staging/worldometer_data/part-00000-65892e24-b897-4a4c-9026-5931b3aa0a78-c000.snappy.parquet"

In [3]:
def read_data(file_path):
    return spark.read.parquet(file_path)

In [4]:
full_grouped_df = read_data(full_grouped_file_path)
# worldometer_df = read_data(worldometer_data_file_path)

In [ ]:
def daily_country_analysis(df: DataFrame) -> DataFrame:
    df = df.groupBy(["Country_Region", "Date"]).agg({"deaths": "sum", "confirmed": "sum"})
    df = df.withColumnsRenamed({"sum(confirmed)": "Confirmed", "sum(deaths)": "Deaths"})
    df = df.filter(df['Confirmed'] != 0)
    df = df.withColumn("Death_Percentage", (df.Deaths / df.Confirmed) * 100)
    return df

In [ ]:
def daily_global_death_analysis(df: DataFrame) -> DataFrame:
    df = df.groupBy("Date").agg({"deaths": "sum", "confirmed": "sum"})
    df = df.withColumnsRenamed({"sum(confirmed)": "Confirmed", "sum(deaths)": "Deaths"})
    df = df.filter(df["Confirmed"] != 0)
    df = df.withColumn("Death_Percentage", (df.Deaths / df.Confirmed) * 100)
    return df

In [ ]:
daily_global_death_analysis(full_grouped_df).take(100)

In [ ]:
worldometer_df.take(1)

In [ ]:
full_grouped_df.take(1)

In [ ]:
def continent_wise_death_analysis(
    full_grouped_df: DataFrame, worldometer_df: DataFrame
) -> DataFrame:
    df = full_grouped_df.join(worldometer_df, "Country_Region")
    df = df.groupBy("Continent").agg({"deaths": "sum", "confirmed": "sum"})
    df = df.withColumnsRenamed({"sum(confirmed)": "Confirmed", "sum(deaths)": "Deaths"})
    df = df.filter(df["Confirmed"] != 0)
    df = df.withColumn("Death_Percentage", (df.Deaths / df.Confirmed) * 100)
    return df

In [ ]:
continent_wise_death_analysis(full_grouped_df, worldometer_df).take(10)

In [ ]:
def country_with_highest_death_percentage(df: DataFrame) -> DataFrame:
    df = df.groupBy(["Country_Region"]).agg({"deaths": "sum", "confirmed": "sum"})
    df = df.withColumnsRenamed({"sum(confirmed)": "Confirmed", "sum(deaths)": "Deaths"})
    df = df.filter(df["Confirmed"] != 0)
    df = df.withColumn("Death_Percentage", (df.Deaths / df.Confirmed) * 100)
    return df.limit(1)

In [ ]:
country_with_highest_death_percentage(full_grouped_df).show()

In [ ]:
from pyspark.sql import functions as sf

def countries_with_most_deaths_per_capita(
    full_grouped_df: DataFrame, worldometer_df: DataFrame
)-> DataFrame:
    df = full_grouped_df.join(worldometer_df, "Country_Region")
    df = df.groupBy(["Country_Region"]).agg({"deaths": "sum", "confirmed": "sum", "population": "first"})
    df = df.withColumnsRenamed({"sum(confirmed)": "Confirmed", "sum(deaths)": "Deaths", "first(population)": "Population"})
    df = df.withColumn("Deaths_Per_Capita",  df.Deaths/ df.Population)
    df = df.sort(df.Deaths_Per_Capita.desc())
    return df.limit(10)

In [ ]:
countries_with_most_deaths_per_capita(full_grouped_df, worldometer_df).show(10, truncate=False)

In [ ]:
def confirmed_cases_per_1000_population(df: DataFrame) -> DataFrame:
    df = df.withColumn(
        "Confirmed_Cases_Per_1000_Population", (df.TotalCases / df.Population) * 1000
    )
    df = df.select('Country_Region','TotalCases','Population','Confirmed_Cases_Per_1000_Population')
    return df

In [ ]:
confirmed_cases_per_1000_population(worldometer_df).show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import col, when

def active_cases_per_1000_population(df: DataFrame) -> DataFrame:
    df = df.withColumn(
        "Active_Cases_Per_1000_Population", 
        when(
            col("Population") > 0,
            (col("ActiveCases") / col("Population")) * 1000
        ).otherwise(None)    
    )
    df = df.select(
        "Country_Region",
        "ActiveCases",
        "Population",
        "Active_Cases_Per_1000_Population",
    )
    return df

In [ ]:
active_cases_per_1000_population(worldometer_df).show(207, truncate=False)

In [ ]:
def top_10_country_with_highest_infection_rate(df: DataFrame) -> DataFrame:
    df = df.sort("Active_Cases_Per_1000_Population", ascending=False)
    df = df.withColumnRenamed("Active_Cases_Per_1000_Population","Infection_Rate")
    return df.limit(10)

In [ ]:
df = active_cases_per_1000_population(worldometer_df)
top_10_country_with_highest_infection_rate(df).show()

In [ ]:
def who_region_ranking(df: DataFrame) -> DataFrame:
    df = df.groupBy(["WHO_Region"]).agg({'Population': 'sum', 'ActiveCases':'sum'})
    df = df.withColumnsRenamed({'sum(ActiveCases)': 'ActiveCases', 'sum(Population)': 'Population'})
    df = df.withColumn(
        'Infection_Rate',
        when(
            col('Population') > 0,
            (col('ActiveCases') / col('Population')) * 1000
        ).otherwise(None)
    )
    return df

In [ ]:
who_region_ranking(worldometer_df).show()

In [ ]:
from pyspark.sql.functions import col, when

def recovered_percentage_per_country(df: DataFrame) -> DataFrame:
    df = df.withColumn(
        "Recovery_Rate",
        when(col("TotalCases") > 0, (col("TotalRecovered") / col("TotalCases")) * 100),
    )
    return df.select("Country_Region", "TotalCases", "TotalRecovered", "Recovery_Rate")

In [ ]:
recovered_percentage_per_country(worldometer_df).show(100)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, avg


def seven_day_recovery_average(df: DataFrame) -> DataFrame:
    window_spec = (
        Window.partitionBy("Country_Region").orderBy("Date").rowsBetween(-6, 0)
    )

    df = df.withColumn(
        "7_day_recovery_average", avg(col("Recovered")).over(window_spec)
    )
    return df

In [ ]:
seven_day_recovery_average(full_grouped_df).show(truncate=False)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, avg, lag, row_number


def daily_recovery_growth(df: DataFrame) -> DataFrame:
    window_spec = Window.partitionBy("Country_Region").orderBy("Date")

    df = df.withColumn("prev_recovered", lag("Recovered", 1).over(window_spec))

    df = df.withColumn(
        "daily_recovery_growth", (col("Recovered") - col("prev_recovered"))
    )

    return df


def average_recovery_growth_per_country(df: DataFrame) -> DataFrame:
    return (
        df.groupBy("Country_Region")
        .agg(avg("daily_recovery_growth").alias("avg_recovery_growth"))
        .orderBy(col("avg_recovery_growth").desc())
    )


def peak_recovery_day_per_country(df: DataFrame) -> DataFrame:
    peak_window = Window.partitionBy("Country_Region").orderBy(
        col("daily_recovery_growth").desc()
    )

    df = df.withColumn("rank", row_number().over(peak_window))

    return df.filter(col("rank") == 1).select(
        "Country_Region", "Date", "daily_recovery_growth"
    )



In [ ]:
    daily_recovery_growth_df = daily_recovery_growth(full_grouped_df)

    average_recovery_growth_per_country_df = average_recovery_growth_per_country(
        daily_recovery_growth_df
    )

    peak_recovery_day_per_country_df = peak_recovery_day_per_country(
        daily_recovery_growth_df
    )

    fatest_recovery_country = average_recovery_growth_per_country_df.limit(1)


In [ ]:
daily_recovery_growth_df.show(10)

In [ ]:
average_recovery_growth_per_country_df.show(10)

In [ ]:
DAY_WISE_FILE_PATH = "hdfs:///data/covid/staging/day_wise/part-00000-878258c0-6596-41f8-b637-87ebe5df7682-c000.snappy.parquet"

In [ ]:
day_wise_df = read_data(DAY_WISE_FILE_PATH)

In [ ]:
day_wise_df.show(1)

In [ ]:
def global_daily_average_new_cases(df: DataFrame) -> DataFrame:
    return df.groupBy("Date").agg(avg("New_cases").alias("Avg_New_Cases"))

In [ ]:
global_daily_average_new_cases(day_wise_df).show(10)

In [ ]:
def detech_spike_days(df: DataFrame)-> DataFrame:
    stats = df.select(
        mean("New_cases").alias("mean_cases"),
        stddev("New_cases").alias("std_cases")
    ).collect()[0]
    
    mean_cases = stats["mean_cases"]
    std_cases = stats["std_cases"]

    df = df.withColumn(
        "z_score",
        (col("New_cases") - mean_cases) / std_cases
    )

    return df.filter(col("z_score") > 2)

In [ ]:
detech_spike_days(day_wise_df).show(100, truncate=False)

In [ ]:
def peak_death_day_globally(df: DataFrame) -> DataFrame:
    df = df.groupby("Date").agg(sum("New_deaths").alias("Total_Deaths"))
    return df.orderBy(col("Total_Deaths").desc()).limit(1)

In [ ]:
peak_death_day_globally(day_wise_df).show()

In [ ]:
def death_growth_rate_month_over_month(df: DataFrame) -> DataFrame:
    df = (
        df.withColumn("Year", year("Date"))
          .withColumn("Month", month("Date"))
    )

    df = (
        df.groupBy("Year", "Month")
        .agg(sum("New_Deaths").alias("Monthly_Deaths"))
        .orderBy("Year","Month")
    )

    window_spec = Window.orderBy("Year", "Month")

    df = df.withColumn(
        "Prev_Month_Deaths",
        lag("Monthly_Deaths").over(window_spec)
    )

    return df.withColumn(
        "MoM_Death_Growth_Rate",
        when(
            col("Prev_Month_Deaths") != 0,
            (
                (col("Monthly_Deaths") - col("Prev_Month_Deaths"))
                / col("Prev_Month_Deaths")
            ) * 100
        )
    )

In [ ]:
death_growth_rate_month_over_month(day_wise_df).show()

In [ ]:
USA_COUNTY_WISE_FILE_PATH = "hdfs:///data/covid/staging/usa_county_wise"

In [ ]:
usa_county_wise_df = read_data(USA_COUNTY_WISE_FILE_PATH)

In [ ]:
usa_county_wise_df.show()

In [ ]:
def aggregate_county_data_to_state_level(df: DataFrame)-> DataFrame:
    return df.groupBy("Date","Province_State").agg(sum("Confirmed").alias("Total_Confirmed"), sum("Deaths").alias("Deaths"))

In [ ]:
aggregate_county_data_to_state_level(usa_county_wise_df).show(100)

In [ ]:
def top_10_affected_states(df: DataFrame) -> DataFrame:
    df = (
        df.groupBy("Province_State")
          .agg(
              sum("Confirmed").alias("Total_Confirmed"),
              sum("Deaths").alias("Total_Deaths")
          )
    )

    return (
        df.orderBy(col("Total_Confirmed").desc())
                .limit(10)
    )

In [ ]:
top_10_affected_states(usa_county_wise_df).show()

In [ ]:
def detech_data_skew_across_state(df: DataFrame) -> DataFrame:
    state_counts = (
        df.groupBy("Province_State")
          .count()
    )

    stats = state_counts.select(
        mean("count").alias("mean"),
        stddev("count").alias("std")
    ).collect()[0]

    mean_val = stats["mean"]
    std_val = stats["std"]
    
    skew_df = state_counts.withColumn(
        "z_score",
        (col("count") - mean_val) / std_val
    )

    return skew_df.filter(col("z_score") > 2)

In [ ]:
detech_data_skew_across_state(usa_county_wise_df).show()

In [ ]:
from pyspark import RDD
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("rdd_based_analysis").getOrCreate()

FULL_GROUPED_FILE_PATH = "hdfs:///data/covid/staging/full_grouped"


def read_data(file_path):
    return spark.read.parquet(file_path)


def total_confirmed_cases_per_country(rdd: RDD) -> RDD:
    country_confirmed_rdd = rdd.map(
        lambda row: (row["Country_Region"], row["Confirmed"])
    )
    return country_confirmed_rdd.reduceByKey(lambda x, y: x + y)


def total_deaths_per_country(rdd: RDD) -> RDD:
    country_deaths = rdd.map(lambda row: (row["Country_Region"], row["Deaths"]))

    return country_deaths.reduceByKey(lambda x, y: x + y)


def death_percentage(rdd: RDD) -> RDD:
    country_stats = rdd.map(
        lambda row: (row["Country_Region"], (row["Confirmed"], row["Deaths"]))
    )

    aggregated_stats = country_stats.reduceByKey(
        lambda x, y: (x[0] + y[0], x[1] + y[1])
    )

    death_percentage = aggregated_stats.mapValues(
        lambda x: (x[0], x[1], (x[1] / x[0]) * 100 if x[0] != 0 else 0)
    )

    return death_percentage.map(lambda row: (row[0], row[1][0], row[1][1], row[1][2])) 

In [ ]:
full_grouped_rdd = read_data(FULL_GROUPED_FILE_PATH)

In [ ]:
full_grouped_rdd = full_grouped_rdd.rdd

In [ ]:
    death_percentage_df = spark.createDataFrame(
        death_percentage(full_grouped_rdd),
        ["Country_Region", "Total_Confirmed", "Total_Deaths", "Death_Percentage"],
    )


In [ ]:
total_confirmed_per_country_df.show()

In [ ]:
    total_deaths_per_country_df = spark.createDataFrame(
        total_deaths_per_country(full_grouped_rdd), ["Country_Region", "Total_Deaths"]
    )


In [ ]:
total_deaths_per_country_df.show()

In [ ]:
    death_percentage_df = spark.createDataFrame(
        death_percentage(full_grouped_rdd),
        ["Country_Region", "Stats"],
    )


In [ ]:
death_percentage_df.show(truncate=False)

In [ ]:
    full_grouped_temp_view = full_grouped_df.createTempView("full_grouped")


In [ ]:
    top_10_infection_countries = spark.sql("""
        SELECT
            Country_Region,
            SUM(Confirmed) AS Total_Confirmed
        FROM full_grouped
        GROUP BY Country_Region
        ORDER BY Total_Confirmed DESC
        LIMIT 10
    """).show()


In [ ]:
    death_percentage_ranking = spark.sql("""
        SELECT
            Country_Region,
            SUM(Confirmed) AS Total_Confirmed,
            SUM(Deaths) AS Total_Deaths,

            CASE
                WHEN SUM(Confirmed) != 0
                THEN (SUM(Deaths) / SUM(Confirmed)) * 100
                ELSE 0
            END AS Death_Percentage

        FROM full_grouped

        GROUP BY Country_Region

        ORDER BY Death_Percentage DESC
    """).show()


In [ ]:
    rolling_7_day_avg = spark.sql("""
        SELECT
            Country_Region,
            Date,
            Confirmed,

            AVG(Confirmed) OVER (
                PARTITION BY Country_Region
                ORDER BY Date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) AS Rolling_7_Day_Avg

        FROM full_grouped
    """).show()


In [7]:
country_partitioned_df = full_grouped_df.repartition("Date", "Country_Region")

In [9]:
country_partitioned_df.show(truncate=False)

+----------+--------------------------------+---------+------+---------+------+---------+----------+-------------+---------------------+
|Date      |Country_Region                  |Confirmed|Deaths|Recovered|Active|New_cases|New_deaths|New_recovered|WHO_Region           |
+----------+--------------------------------+---------+------+---------+------+---------+----------+-------------+---------------------+
|2020-01-22|Qatar                           |0        |0     |0        |0     |0        |0         |0            |Eastern Mediterranean|
|2020-01-24|West Bank and Gaza              |0        |0     |0        |0     |0        |0         |0            |Eastern Mediterranean|
|2020-01-27|Iceland                         |0        |0     |0        |0     |0        |0         |0            |Europe               |
|2020-01-28|Afghanistan                     |0        |0     |0        |0     |0        |0         |0            |Eastern Mediterranean|
|2020-01-28|Burkina Faso                 